# Importing Necessary python libraries and Dataset

In [46]:
!pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as mtp

In [3]:
data=pd.read_csv(r'E:\Projects\Python\Customer Shopping Behaviour\customer_shopping_behavior.csv')

# Description of the Dataset

In [4]:
data.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [6]:
data.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [7]:
data.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

# Data Transformation

In [8]:
data['Review Rating']=data.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median() ))

In [9]:
data.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [10]:
data.columns=data.columns.str.lower()
data.columns=data.columns.str.replace(' ','_')
data=data.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [11]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [12]:
labels = ['Young_Adults','Adult','Middle_aged', 'Senior']
data['age_group']= pd.qcut(data['age'], q=4, labels=labels)

In [13]:
data[['age','age_group']].head(5)

,age,age_group
0,55,Middle_aged
1,19,Young_Adults
2,50,Middle_aged
3,21,Young_Adults
4,45,Middle_aged


In [14]:
#Creating purchase_frequency_days
frequency_mapping= {
    'Fortnightly':14,
    'Weekly':7,
    'Monthly':30,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Annually':365,
    'Every_3_months':90 
}
data['purchase_frequency_days']=data['frequency_of_purchases'].map(frequency_mapping)

In [15]:
data[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14.0,Fortnightly
1,14.0,Fortnightly
2,7.0,Weekly
3,7.0,Weekly
4,365.0,Annually
5,7.0,Weekly
6,90.0,Quarterly
7,7.0,Weekly
8,365.0,Annually
9,90.0,Quarterly


In [16]:
data['frequency_of_purchases'].unique()

<StringArray>
[   'Fortnightly',         'Weekly',       'Annually',      'Quarterly',
      'Bi-Weekly',        'Monthly', 'Every 3 Months']
Length: 7, dtype: str

In [17]:
(data[ 'discount_applied']==data['promo_code_used']).all()

np.True_

In [18]:
data=data.drop('promo_code_used', axis=1)

In [19]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

# Database Setup & Data Loading

In [20]:
from sqlalchemy import  create_engine

In [21]:
usar = "postgres"
password = "postgresql"
host = "localhost"
port = "5432"
database = "customer_db"

engine = create_engine(
    f"postgresql+psycopg2://{usar}:{password}@{host}:{port}/{database}"
)
table_name= "customer"
data.to_sql(table_name, engine, if_exists= "replace", index= False)
print(f"Data successfully loaded into table: '{table_name}' in database '{database}'.")

Data successfully loaded into table: 'customer' in database 'customer_db'.


# Business Questions & SQL Analysis

### 1. Revenue Contribution by Male and Female

In [22]:
query = """
SELECT
gender,
SUM(purchase_amount) AS Revenue
FROM customer
GROUP BY gender; 
"""
result = pd.read_sql(query, engine)
result

,gender,revenue
0,Female,75191.0
1,Male,157890.0


### 02. Which customers used discount and still spent more than the average

In [23]:
query = """SELECT
customer_id,
SUM(purchase_amount) AS revenue,
(SELECT ROUND(AVG(purchase_amount), 2)FROM customer) AS avg_purchase
FROM customer
WHERE discount_applied = 'Yes'
GROUP BY customer_id
HAVING SUM(purchase_amount) > (SELECT AVG(purchase_amount)FROM customer)
ORDER BY revenue DESC;
"""
result = pd.read_sql(query, engine)
result

,customer_id,revenue,avg_purchase
0,96,100.0,59.76
1,616,100.0,59.76
2,582,100.0,59.76
3,1592,100.0,59.76
4,194,100.0,59.76
...,...,...,...
834,677,60.0,59.76
835,1002,60.0,59.76
836,635,60.0,59.76
837,589,60.0,59.76


### 03. Top 5 products with highest average review ratings

In [24]:
query = """SELECT
item_purchased,
category,
ROUND(AVG(review_rating)::numeric,2) AS avg_rating
FROM customer
GROUP BY item_purchased, category
ORDER BY avg_rating DESC
LIMIT 5
"""
result = pd.read_sql(query, engine)
result

,item_purchased,category,avg_rating
0,Gloves,Accessories,3.86
1,Sandals,Footwear,3.84
2,Boots,Footwear,3.82
3,Hat,Accessories,3.80
4,Handbag,Accessories,3.78


### 04. Compare the average purchase amount between standered and express Shipping

In [25]:
query = """SELECT
shipping_type,
ROUND(AVG(purchase_amount):: numeric,2) AS avg_purchase_amount
FROM customer
WHERE "shipping_type" IN('Standard','Express')
GROUP BY shipping_type
"""
result = pd.read_sql(query, engine)
result

,shipping_type,avg_purchase_amount
0,Express,60.48
1,Standard,58.46


### 05. Do subscribed customers spend more? Compare average spend and total revenue between subscibers and non subscribers

In [26]:
query = """SELECT
subscription_status,
COUNT(customer_id) AS total_customer,
SUM(purchase_amount) AS total_revenue,
ROUND (AVG(purchase_amount)::numeric,2) AS avg_revenue
FROM customer
GROUP BY subscription_status
ORDER BY total_revenue DESC
"""
result = pd.read_sql(query, engine)
result

,subscription_status,total_customer,total_revenue,avg_revenue
0,No,2847,170436.0,59.87
1,Yes,1053,62645.0,59.49


### 06. Which 5 products have the highest percentage of purchase with discount applied?

In [27]:
query = """SELECT
item_purchased,
ROUND (100* SUM(CASE WHEN "discount_applied"='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) AS discount_rate
FROM customer
GROUP BY item_purchased
"""
result = pd.read_sql(query, engine)
result

,item_purchased,discount_rate
0,Hoodie,45.0
1,Dress,45.0
2,Boots,46.0
3,Scarf,42.0
4,Blouse,33.0
5,Shirt,42.0
6,Jeans,45.0
7,Sweater,48.0
8,Sunglasses,40.0
9,Gloves,42.0


### 07. Segment customers based on the number of previous purchases in New, Returning and loyal and Show number in each segmentation

In [28]:
query = """WITH customer_segmantation AS
(
SELECT
"customer_id", 
"previous_purchases",
(
CASE 
WHEN "previous_purchases"= 1 THEN 'New'
WHEN "previous_purchases" BETWEEN 2 AND 10 THEN 'Returning'
ELSE 'Loyal'
END
) AS customer_category
FROM customer
)
SELECT
"customer_category",
COUNT ("customer_id") AS "no_of_customers"
FROM customer_segmantation
GROUP BY "customer_category"
"""
result = pd.read_sql(query, engine)
result

,customer_category,no_of_customers
0,New,83
1,Returning,701
2,Loyal,3116


### 08. What are the most 3 purchahed products within each category

In [31]:
query = """WITH category_wise_product AS
(
SELECT
"item_purchased",
"category",
COUNT("customer_id") AS "total_Orders",
ROW_NUMBER () OVER (PARTITION BY "category" ORDER BY COUNT("customer_id") DESC) AS "category_rank"
FROM customer
GROUP BY "item_purchased", "category"
)
SELECT
"category_rank",
"category",
"item_purchased",
"total_Orders"
FROM category_wise_product WHERE "category_rank" <=3
"""
result = pd.read_sql(query, engine)
result

,category_rank,category,item_purchased,total_Orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Belt,161
2,3,Accessories,Sunglasses,161
3,1,Clothing,Blouse,171
4,2,Clothing,Pants,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


### 09. Are customers are repeated buyers (more than 5 times) who likely to subscribe?

In [32]:
query = """SELECT 
"subscription_status",
COUNT ("customer_id") AS "repeated_customers"
FROM customer 
WHERE "previous_purchases" >5
GROUP BY "subscription_status"
"""
result = pd.read_sql(query, engine)
result

,subscription_status,repeated_customers
0,No,2518
1,Yes,958


### 10. What is the revenue contributiuon of each age group 

In [33]:
query = """SELECT
"age_group",
SUM("purchase_amount") AS total_revenue
FROM customer
GROUP BY "age_group"
"""
result = pd.read_sql(query, engine)
result

,age_group,total_revenue
0,Young_Adults,62143.0
1,Adult,55978.0
2,Middle_aged,59197.0
3,Senior,55763.0
